# Explicit + PCA topic embeddings

1. Count comments per topic and author across the entire input, reading comments in chunks.
2. Mark presence when the total is **strictly greater than 5**.
3. Count shared users between topics, zero the diagonal, and drop topics without any overlap.
4. Choose **500** retained context topics by users above the cutoff; ties follow topic-file order.
5. Add one to every context-filtered count, compute natural-log PPMI.
6. Fit centered, non-whitened **PCA with 150 components** to the row-normalized 500-dimensional PPMI vectors. Always normalize the reduced vectors for cosine-based evaluation.

Outputs go to `my_data/explicit/`. 

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from IPython.display import display

DATA_DIR = Path("my_data")
ARTICLE_PATH = DATA_DIR / "articles_top_1000.csv"
RAW_DATA_DIR = Path("data")
COMMENT_PATH = DATA_DIR / "comments_top_1000.csv"
TOPIC_PATH = DATA_DIR / "topics_top_1000.csv"
OUTPUT_DIR = DATA_DIR / "explicit"
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
USE_RAW_COMMENTS = False
CHUNK_SIZE = 500_000
CUTOFF = 5
CONTEXT_DIMENSIONS = 500
PCA_DIMENSIONS = 150

## 1. Load topic and article mappings

Validate IDs and paths before counting. Blank authors and comments whose articles are absent from the supplied article file are excluded and counted in the run metadata. Each remaining comment row contributes one count, as in the reference notebook.

In [2]:
topics = pd.read_csv(TOPIC_PATH, keep_default_na=False, dtype=str)
articles = pd.read_csv(ARTICLE_PATH, keep_default_na=False, dtype=str,
                       usecols=["Article_Id", "Topic_Id", *TOPIC_COLUMNS])
topic_to_index = {topic_id: i for i, topic_id in enumerate(topics["Topic_Id"])}
article_to_index = articles.set_index("Article_Id")["Topic_Id"].map(topic_to_index)

def comment_chunks():
    if USE_RAW_COMMENTS:
        paths = sorted(RAW_DATA_DIR.glob("comments_file_*_[0-9][0-9].tsv"))
        if not paths:
            raise FileNotFoundError("No raw comment shards found")
    else:
        paths = [COMMENT_PATH]
    for path in paths:
        with pd.read_csv(path, sep="\t" if USE_RAW_COMMENTS else ",",
                         keep_default_na=False, dtype=str,
                         usecols=["Article_Id", "Author_Id"],
                         chunksize=CHUNK_SIZE) as reader:
            yield from reader

## 2. Accumulate sparse topic–user counts

Count number of comments per user on each topic

In [3]:
def count_topic_users(comment_chunks, article_mapping, n_topics, verbose=False):
    """Accumulate full-dataset counts; return sparse counts and processing statistics."""
    users = {}
    counts = csr_matrix((n_topics, 0), dtype=np.int64)
    stats = dict(comment_rows=0, valid_comments=0, excluded_unknown_articles=0,
                 excluded_blank_authors=0)
    for chunk_number, chunk in enumerate(comment_chunks, start=1):
        topic_indices = chunk["Article_Id"].map(article_mapping)
        authors = chunk["Author_Id"].str.strip()
        known = topic_indices.notna()
        nonblank = authors.ne("")
        valid = known & nonblank
        stats["comment_rows"] += len(chunk)
        stats["excluded_unknown_articles"] += int((~known).sum())
        stats["excluded_blank_authors"] += int((known & ~nonblank).sum())
        stats["valid_comments"] += int(valid.sum())
        valid_authors = authors.loc[valid]
        for author in valid_authors.unique():
            if author not in users:
                users[author] = len(users)
        counts.resize((n_topics, len(users)))
        chunk_counts = csr_matrix(
            (np.ones(int(valid.sum()), dtype=np.int64),
             (topic_indices.loc[valid].to_numpy(dtype=np.int64),
              valid_authors.map(users).to_numpy(dtype=np.int64))),
            shape=counts.shape,
        )
        counts = counts + chunk_counts
        if verbose and chunk_number % 10 == 0:
            print(f"Processed {stats['comment_rows']:,} comments; {len(users):,} authors.", flush=True)
    if counts.sum() != stats["valid_comments"]:
        raise AssertionError("Sparse counts do not match the valid-comment total.")
    stats.update(users=len(users), topic_user_relationships=counts.nnz)
    return counts, stats

topic_user_counts, counting_stats = count_topic_users(
    comment_chunks(), article_to_index, len(topics), verbose=True
)
print(json.dumps(counting_stats, indent=2))
print(f"Topic–user matrix shape: {topic_user_counts.shape}")

Processed 5,000,000 comments; 72,255 authors.
Processed 10,000,000 comments; 91,664 authors.
Processed 15,000,000 comments; 106,734 authors.
Processed 20,000,000 comments; 120,925 authors.
Processed 25,000,000 comments; 132,821 authors.
Processed 30,000,000 comments; 142,556 authors.
Processed 35,000,000 comments; 151,711 authors.
Processed 40,000,000 comments; 160,490 authors.
Processed 45,000,000 comments; 170,185 authors.
{
  "comment_rows": 48871807,
  "valid_comments": 48871807,
  "excluded_unknown_articles": 0,
  "excluded_blank_authors": 0,
  "users": 176892,
  "topic_user_relationships": 5556802
}
Topic–user matrix shape: (1000, 176892)


## 2. Shared-user co-occurrence and context selection

An author contributes at most one presence per topic, after exceeding the cutoff. The diagonal is removed before selecting the context columns. Topics without overlap are explicitly recorded; context topics are chosen only from retained topics.

In [4]:
def build_context_counts(counts, cutoff=5, context_dimensions=500):
    presence = (counts > cutoff).astype(np.int64)
    active_users = np.asarray(presence.sum(axis=1)).ravel()
    cooccurrence = (presence @ presence.T).tocsr()
    assert np.array_equal(cooccurrence.diagonal(), active_users)
    cooccurrence.setdiag(0)
    cooccurrence.eliminate_zeros()
    retained_indices = np.flatnonzero(np.asarray(cooccurrence.sum(axis=1)).ravel() > 0)
    if len(retained_indices) < context_dimensions:
        raise ValueError(f"Need {context_dimensions} overlapping topics; found {len(retained_indices)}.")
    context_indices = retained_indices[
        np.argsort(-active_users[retained_indices], kind="stable")[:context_dimensions]
    ]
    context_counts = cooccurrence[retained_indices][:, context_indices].toarray().astype(np.float64)
    return context_counts, retained_indices, context_indices, active_users

context_counts, retained_indices, context_indices, active_users = build_context_counts(
    topic_user_counts, cutoff=CUTOFF, context_dimensions=CONTEXT_DIMENSIONS
)
retained_topics = topics.iloc[retained_indices][["Topic_Id", *TOPIC_COLUMNS]].reset_index(drop=True)
dropped_topics = topics.loc[~topics.index.isin(retained_indices), ["Topic_Id", *TOPIC_COLUMNS]].copy()
dropped_topics["reason"] = "No shared users with another topic after cutoff"
print(f"Retained {len(retained_topics)} topics; dropped {len(dropped_topics)} without overlap.")
print(f"Context count matrix: {context_counts.shape}")
print(f"Topic–user presences above cutoff: {(topic_user_counts > CUTOFF).nnz:,}")
display(dropped_topics)

Retained 1000 topics; dropped 0 without overlap.
Context count matrix: (1000, 500)
Topic–user presences above cutoff: 1,288,182


,Topic_Id,Topic_1,Topic_2,Topic_3,reason


## 3. Smoothed PPMI in 500 dimensions

Apply add-one smoothing to **every entry**, including the zeroed self-pairs, matching the reference. Marginals and total mass are computed from this smoothed, context-filtered matrix: `PPMI(i,j) = max(log(count(i,j) * total / (row_total(i) * column_total(j))), 0)`.

PPMI rows are always normalized to unit length before PCA so PCA does not account for vector lenght. Zero rows are reported and retained so row alignment is preserved.

In [5]:
def normalize_rows(values):
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    normalized = np.divide(values, norms, out=np.zeros_like(values), where=norms > 0)
    return normalized, np.flatnonzero(norms.ravel() == 0)


def compute_ppmi(context_counts):
    smoothed = context_counts + 1.0
    total = smoothed.sum()
    row_totals = smoothed.sum(axis=1, keepdims=True)
    column_totals = smoothed.sum(axis=0, keepdims=True)
    return np.maximum(np.log(smoothed * total / (row_totals * column_totals)), 0.0)

ppmi = compute_ppmi(context_counts)
zero_ppmi_rows = np.flatnonzero(np.linalg.norm(ppmi, axis=1) == 0)
assert ppmi.shape == (len(retained_topics), CONTEXT_DIMENSIONS)
assert np.isfinite(ppmi).all()
print(f"Explicit embeddings: {ppmi.shape}; zero PPMI rows: {len(zero_ppmi_rows)}")

Explicit embeddings: (1000, 500); zero PPMI rows: 0


## 4. PCA: 500 → 150 dimensions

PCA centers the row-normalized PPMI features, uses full SVD for reproducibility, and does not whiten. Save both the raw PCA coordinates and their unit-length versions; negative coordinates after PCA are expected. Explained variance measures the variance retained before the final normalization.

In [6]:
pca_input, _ = normalize_rows(ppmi)

if PCA_DIMENSIONS > min(pca_input.shape):
    raise ValueError("PCA dimensions exceed the available topics or context dimensions.")
pca = PCA(n_components=PCA_DIMENSIONS, svd_solver="full", whiten=False)
topic_embeddings = pca.fit_transform(pca_input)
normalized_topic_embeddings, zero_pca_rows = normalize_rows(topic_embeddings)
if len(zero_pca_rows):
    raise ValueError("PCA produced zero vectors; cosine evaluation would be undefined.")
assert topic_embeddings.shape == (len(retained_topics), PCA_DIMENSIONS)
assert np.isfinite(topic_embeddings).all()
assert np.allclose(np.linalg.norm(normalized_topic_embeddings, axis=1), 1.0)
explained_variance = pd.DataFrame({
    "component": np.arange(1, PCA_DIMENSIONS + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
})
print(f"Reduced embeddings: {topic_embeddings.shape}")
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.2%}")
display(explained_variance.iloc[[0, -1]])

Reduced embeddings: (1000, 150)
Variance retained: 95.47%


,component,explained_variance_ratio,cumulative_explained_variance_ratio
0,1,0.169881,0.169881
149,150,0.000328,0.954653


## 5. Export embeddings and reproducibility metadata

- `topic_embeddings.csv`: raw 150-dimensional PCA coordinates.
- `topic_embeddings_normalized.csv`: unit-length 150-dimensional coordinates for evaluation.
- `topic_embeddings_500_ppmi.csv`: unnormalized PPMI before PCA.
- `context_topics.csv`: maps each original embedding column to its context topic.
- `pca_explained_variance.csv`, `dropped_topics.csv`, and `run_metadata.json`: diagnostics and configuration.

In [ ]:
def embedding_frame(values):
    return pd.concat([
        retained_topics,
        pd.DataFrame(values, columns=[f"embedding_{i:03d}" for i in range(values.shape[1])]),
    ], axis=1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
embedding_frame(topic_embeddings).to_csv(OUTPUT_DIR / "topic_embeddings.csv", index=False)
embedding_frame(normalized_topic_embeddings).to_csv(OUTPUT_DIR / "topic_embeddings_normalized.csv", index=False)
embedding_frame(ppmi).to_csv(OUTPUT_DIR / "topic_embeddings_500_ppmi.csv", index=False)
context_topics = topics.iloc[context_indices][["Topic_Id", *TOPIC_COLUMNS]].reset_index(drop=True)
context_topics.insert(0, "embedding_column", [f"embedding_{i:03d}" for i in range(CONTEXT_DIMENSIONS)])
context_topics["users_above_cutoff"] = active_users[context_indices]
context_topics.to_csv(OUTPUT_DIR / "context_topics.csv", index=False)
dropped_topics.to_csv(OUTPUT_DIR / "dropped_topics.csv", index=False)
run_metadata = dict(
    reference_notebook="0_explicit_embeddings_18.ipynb",
    articles=str(ARTICLE_PATH), comments=str(COMMENT_PATH), topics=str(TOPIC_PATH),
    scope="All rows and years in the supplied top-1000 files", cutoff=CUTOFF,
    presence_rule="comments > cutoff", context_dimensions=CONTEXT_DIMENSIONS,
    smoothing=1.0, pmi_log="natural", pca_input="row-normalized PPMI",
    normalize_before_pca=True,
    dimensions=PCA_DIMENSIONS, pca_solver="full", pca_whiten=False,
    input_topics=len(topics), retained_topics=len(retained_topics), dropped_topics=len(dropped_topics),
    zero_ppmi_topic_ids=retained_topics.iloc[zero_ppmi_rows]["Topic_Id"].tolist(),
    explained_variance_ratio=float(pca.explained_variance_ratio_.sum()), **counting_stats,
)
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
print(f"Saved embeddings for {len(retained_topics)} topics to {OUTPUT_DIR.resolve()}")
print(f"Dimensions: {CONTEXT_DIMENSIONS} → {PCA_DIMENSIONS}; variance retained: {pca.explained_variance_ratio_.sum():.2%}")
display(embedding_frame(normalized_topic_embeddings).head())

Saved embeddings for 1000 topics to /Users/nachodoradollamas/Downloads/derStandard/my_data/explicit
Dimensions: 500 → 150; variance retained: 95.47%


,Topic_Id,Topic_1,Topic_2,Topic_3,embedding_000,embedding_001,embedding_002,embedding_003,embedding_004,embedding_005,...,embedding_140,embedding_141,embedding_142,embedding_143,embedding_144,embedding_145,embedding_146,embedding_147,embedding_148,embedding_149
0,0,diskurs,debatten,kommentare-der-anderen,0.093926,-0.260758,-0.106968,0.475649,-0.153824,0.618877,...,-0.021484,0.038078,0.014669,0.003668,0.015170,-0.006296,0.005383,-0.010271,-0.009659,0.019667
1,1,diskurs,kolumnen,rauscher,0.088416,-0.271908,-0.098130,0.470057,-0.194442,0.620173,...,-0.020175,0.007612,0.010033,-0.004913,0.014992,-0.008029,0.009250,-0.004031,-0.005604,0.011027
2,2,international,amerika,usa,0.171992,-0.290378,-0.027116,0.468272,-0.058009,0.577220,...,-0.029310,-0.012058,0.004970,0.007337,-0.013589,0.006845,0.020737,-0.004556,-0.027026,-0.000160
3,3,international,europa,deutschland,0.098888,-0.315483,-0.012028,0.465419,-0.144462,0.628314,...,-0.012201,0.010504,0.013385,-0.000589,0.007592,-0.015980,0.016163,-0.000543,0.000397,-0.006249
4,4,inland,parteien,fpoe,0.183785,-0.282750,0.008749,0.476713,-0.269633,0.595711,...,-0.001931,0.027502,0.005254,0.003042,-0.006401,-0.022004,-0.009443,-0.009703,0.013727,-0.022050
